In [1]:
%pip install lark

Note: you may need to restart the kernel to use updated packages.


In [2]:
from lark import Lark

dsl_grammar = r"""
    ?start: doctag

    doctag: "<doctag>" top_level "</doctag>"

    ?top_level: list | group | semantic
    list: "<list>" list_item "</list>"
    list_item: "<list_item>" semantic "</list_item>"

    group: "<group>" semantic "</group>"

    ?semantic: title | section_header | text | caption | formatting
    title: "<title>" [brect] formatting "</title>"
    section_header: "<section_header>" [brect] formatting "</section_header>"
    text: "<text>" [brect] formatting "</text>"
    caption: "<caption>" [brect] formatting "</caption>"

    ?formatting: bold | italic | verbatim
    bold: "<bold>" formatting "</bold>"
    italic: "<italic>" formatting "</italic>"

    ?verbatim: code | formula | raw_text

    code: "<code lang='" language_name "'>" [brect] raw_text "</code>"
        | "<code>" [language] [brect] raw_text "</code>"

    formula: "<formula>" [brect] raw_text "</formula>"

    ?brect: location location location location
    ?location: "<location_" LOCATION_VALUE "/>"

    ?language: "<" language_name "/>"
    language_name: LANGUAGE_NAME

    raw_text: (TEXT | OTHER_TAG)*

    LOCATION_VALUE: /0|[1-9][0-9]*/

    LANGUAGE_NAME: /(python|java)/

    TEXT:       /[^<]+/
    OTHER_TAG:  /<[^>]*>?/
"""


In [3]:
tests = {
    "<doctag><group><title>My title here</title></group></doctag>": "group",
    "<doctag><list><list_item>Single item</list_item></list></doctag>": "list",
    "<doctag><list><list_item><italic>Single italic item</italic></list_item></list></doctag>": "list",
    # "<doctag><group><list_item><italic>Single italic item</italic></list_item></group></doctag>": "group",  # TODO: here list item is interpreted as raw_text, could be a feature
    # "<doctag><list><st item</list_item><list_item><italic>Second item</italic></list_item></list></doctag>": "list",  # TODO: fix this

    "<doctag><title><location_12/><location_42/><location_24/><location_19/>Title</title></doctag>": "title",
    "<doctag><section_header>This is my section</section_header></doctag>": "section_header",
    "<doctag><text>My text</text></doctag>": "text",
    "<doctag><caption>Caption</caption></doctag>": "caption",
    "<doctag><title><italic>Welcome</italic></title></doctag>": "title",
    # "<doctag><title>hi <italic>Welcome</italic></title></doctag>": "title",  # TODO: fix this

    "<doctag><bold>foo</bold></doctag>": "bold",
    "<doctag><italic>bar</italic></doctag>": "italic",
    "<doctag><bold><italic>baz</italic></bold></doctag>": "bold",
    "<doctag><bold><code><python/>print('Hello')</code></bold></doctag>": "bold",

    "<doctag><code>This is some code.</code></doctag>": "code",
    "<doctag><code>This is more code.\nIt has two lines.</code></doctag>": "code",
    "<doctag><code></code></doctag>": "code",
    "<doctag><code>print('Hello world')</code></doctag>": "code",
    "<doctag><code><python/>print('Hello')</code></doctag>": "code",
    "<doctag><code><java/><location_12/><location_42/><location_24/><location_19/>system.out.println(\"Hello\")</code></doctag>": "code",
    "<doctag><code>system.out.println(\"Hello\")<java/></code></doctag>": "code",  # TODO: fix this
    "<doctag><code lang='python'>Hi</code></doctag>": "code",
    # "<doctag><code><javascript/>console.log('unsupported')</code></doctag>": "code",  # TODO: fix this: should still be code
    "<doctag><code>Tag mismatch</formula></doctag>": "raw_text",

    "<doctag><formula>This is some formula.</formula></doctag>": "formula",
    "<doctag><formula></formula></doctag>": "formula",

    "<doctag>This is just regular text.</doctag>": "raw_text",
    # "<doctag><other>Undefined tag</other></doctag>": "raw_text",  # TODO: fix this
    # "<doctag><unbalanced</doctag>": "raw_text",  # TODO: fix this
    "<doctag></doctag>": "raw_text",

    # "<doctag>foo<page_break/>bar</doctag>": "raw_text",  # TODO: fix this 👈

}


In [4]:
parser = Lark(dsl_grammar)

for text, expected_type in tests.items():
    print(f"{80*'-'}\n{text=}:")
    parsed_tree = parser.parse(text)
    print(parsed_tree.pretty().rstrip())
    assert (d := parsed_tree.pretty().lstrip()).startswith("doctag") and d.removeprefix("doctag").lstrip().startswith(expected_type)


--------------------------------------------------------------------------------
text='<doctag><group><title>My title here</title></group></doctag>':
doctag
  group
    title
      None
      raw_text	My title here
--------------------------------------------------------------------------------
text='<doctag><list><list_item>Single item</list_item></list></doctag>':
doctag
  list
    list_item
      raw_text	Single item
--------------------------------------------------------------------------------
text='<doctag><list><list_item><italic>Single italic item</italic></list_item></list></doctag>':
doctag
  list
    list_item
      italic
        raw_text	Single italic item
--------------------------------------------------------------------------------
text='<doctag><title><location_12/><location_42/><location_24/><location_19/>Title</title></doctag>':
doctag
  title
    brect
      12
      42
      24
      19
    raw_text	Title
----------------------------------------------------------